# Code Distillation: Natural Language to Python

This notebook's primary mode reproduces the **explicitly documented historical winning recipe** from the original `Demos/NL_to_Python_Distillation` demo. It requests 2,000 live examples, applies the original four-dimension quality and syntax gates, deterministically selects exactly 1,500 training rows and 84 zero-overlap held-out rows, fine-tunes GPT-4.1-mini for one epoch, and measures correctness/conciseness gain on all 84 held-out prompts.

Live generation is the only primary path: the exact generated bytes are filtered, split, hashed, uploaded, and trained. There is no fallback to committed snapshot data.


## 1. Fixed compatibility contract

| Surface | Parity value |
|---|---|
| Teacher/generator | GPT-5.4 project deployment |
| Student/base | `gpt-4.1-mini-2025-04-14` and matching base deployment |
| Evaluator | GPT-5.4, independent scoring calls at temperature 0 |
| Requested generation | 2,000 rows |
| Accepted cohort | At least 1,584 rows after quality, syntax, and deduplication gates |
| Train/validation/evaluation | 1,500 train; 84 held out; the same 84 are training validation and independent quality evaluation |
| Quality gate | Mean Relevance, Pythonic, Readability, Efficiency >= 3.0/4 plus valid Python syntax |
| Hyperparameters | Supervised; 1 epoch; LR multiplier 1.3; batch size 1 |
| Primary metric | Mean correctness and conciseness (1-10), their combined mean, combined Pass@8 |
| Gain | `(fine_tuned_combined - base_combined) / base_combined * 100` using unrounded 84-row means |

The original source also contains incompatible 11/1 and approximately 85/4 small runs. Its prose says a 5% holdout and 70-80% acceptance, which does not mathematically yield exactly 1,500 train plus 84 held out. The explicit historical winning counts are therefore the parity target. The old code also computed Pass@8 from correctness only despite describing combined Pass@8; both are recorded, with combined Pass@8 primary.


In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
DEMO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
load_dotenv(DEMO_ROOT / ".env")

required = [
    "FOUNDRY_PROJECT_ENDPOINT",
    "FOUNDRY_TEACHER_DEPLOYMENT",
    "FOUNDRY_STUDENT_DEPLOYMENT",
    "FOUNDRY_EVALUATOR_DEPLOYMENT",
]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Configure {missing} in {DEMO_ROOT / '.env'}")

visible_config = {
    "project_endpoint": os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    "teacher": os.environ["FOUNDRY_TEACHER_DEPLOYMENT"],
    "student_base": os.environ["FOUNDRY_STUDENT_DEPLOYMENT"],
    "evaluator": os.environ["FOUNDRY_EVALUATOR_DEPLOYMENT"],
    "student_model": os.environ.get("FOUNDRY_STUDENT_MODEL", "gpt-4.1-mini-2025-04-14"),
    "requested": int(os.environ.get("FOUNDRY_NUM_RECORDS", "2000")),
    "train": int(os.environ.get("FOUNDRY_TRAIN_RECORDS", "1500")),
    "held_out": int(os.environ.get("FOUNDRY_EVALUATION_RECORDS", "84")),
    "quality_threshold": float(os.environ.get("FOUNDRY_QUALITY_THRESHOLD", "3.0")),
}
print(json.dumps(visible_config, indent=2))


## 2. Original generation and filtering method

The runner uses the original hierarchical steering columns: industry sector and topic, complexity and code concept, and instruction phrase. GPT-5.4 generates diverse instructions at temperature 1.0 and code at temperature 0.4. The same teacher role judges Relevance, Pythonic style, Readability, and Efficiency on the original 0-4 rubrics. Rows must average at least 3.0, pass Python AST validation, and survive normalized-instruction deduplication.

After a seed-42 shuffle, parity mode takes the first 1,500 accepted rows for training and the next 84 as the held-out validation/evaluation set. Surplus accepted rows are retained in lineage evidence but excluded to keep the historical winning scale exact.


In [ ]:
sys.path.insert(0, str(DEMO_ROOT))
import live_validate as parity

assert visible_config["requested"] == parity.PARITY_GENERATED_COUNT == 2000
assert visible_config["train"] == parity.PARITY_TRAIN_COUNT == 1500
assert visible_config["held_out"] == parity.PARITY_EVALUATION_COUNT == 84
assert visible_config["quality_threshold"] == parity.PARITY_QUALITY_THRESHOLD == 3.0
print("Exact historical parity scale and method are locked.")


## 3. Training and primary gain measurement

The uploaded files are byte-hash checked immediately before submission. A prior fine-tuning job may be reused only when the project, student model, exact train and validation hashes, counts, one epoch, LR multiplier 1.3, and batch size 1 all match.

Teacher, base student, and fine-tuned student then answer the same 84 held-out prompts with the same system prompt and temperature 0. GPT-5.4 independently assigns correctness and conciseness integer scores from 1 to 10 using the original rubric. The report calculates combined means, combined Pass@8, legacy correctness-only Pass@8, relative combined gain, and code-length change.


## 4. Secondary sandbox diagnostic and safety boundary

The existing four-case `APP_MODE` behavioral check remains secondary; it does not replace the original 84-row correctness/conciseness measurement. Candidate code is **never executed in this notebook kernel or its host Python process**. The runner submits it to a Foundry-hosted code-interpreter container and records the execution boundary and per-case results. Local handling is limited to text processing and AST syntax parsing.


In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, str(DEMO_ROOT / "live_validate.py")],
    cwd=DEMO_ROOT,
    check=True,
    text=True,
)
print("Live parity workflow completed with exit code", completed.returncode)


In [ ]:
reports = sorted((DEMO_ROOT / "outputs").glob("*/parity-report.json"))
if not reports:
    raise RuntimeError("No parity report was produced.")
report_path = reports[-1]
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Report:", report_path)
print(json.dumps({
    "job": report["job"],
    "lineage": report["lineage"],
    "training_metrics": report["training_metrics"],
    "primary_evaluation": report["primary_evaluation"],
    "gain": report["gain"],
    "sandbox": report["sandbox"],
}, indent=2))


## 5. Interpretation

A successful job is not by itself a parity success. The verdict must use all 84 original-compatible held-out evaluations and separately disclose the hosted sandbox result. Compare the new unrounded relative gain with the historical reported +6.4%, while noting that the published rounded 8.6-to-9.2 scores imply about +7.0% and therefore cannot reconstruct the exact historical percentage.

All runtime identifiers, live datasets, hashes, upload records, job responses, predictions, and evaluation evidence remain under ignored `outputs/`.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.
